# Gold:
## dbt checks

In [0]:
%sql
-- 1) Nutri-Score dim (should be exactly 6 seed rows: a..e, unknown)
SELECT grade, COUNT(*) AS n
FROM nutrichain_lakehouse.gold.dim_nutriscore
GROUP BY grade
ORDER BY grade;

-- Compare to what seed should have loaded
SELECT grade
FROM nutrichain_lakehouse.gold.nutriscore_grade_lookup
ORDER BY grade;

-- 2) Data quality tier on products
SELECT data_quality_tier, COUNT(*) AS n
FROM nutrichain_lakehouse.gold.dim_product
GROUP BY data_quality_tier
ORDER BY n DESC;

-- 3) Sugar tier at source (Silver) — this is what you "control" via PySpark
SELECT sugar_tier, COUNT(*) AS n
FROM nutrichain_lakehouse.silver.silver_openfood_products
GROUP BY sugar_tier
ORDER BY n DESC;

-- 4) When Gold was last built (stale table smell test)
SELECT MAX(gold_built_at) AS last_gold_build
FROM nutrichain_lakehouse.gold.dim_product;

In [0]:
%sql
SELECT COUNT(*) FROM nutrichain_lakehouse.gold.fact_product_nutrition;
SELECT COUNT(*) FROM nutrichain_lakehouse.gold.dim_product;

## % rows with energy populated - Fact table

In [0]:
%sql
SELECT
    COUNT(*) AS fact_rows,
    ROUND(100.0 * SUM(CASE WHEN energy_kcal_per_100g IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2)
        AS energy_kcal_populated_pct
FROM nutrichain_lakehouse.gold.fact_product_nutrition;

## Fact completeness

In [0]:
%sql
SELECT
  COUNT(*) AS fact_rows,
  ROUND(100.0 * SUM(CASE WHEN energy_kcal_per_100g IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS energy_populated_pct
FROM nutrichain_lakehouse.gold.fact_product_nutrition;

##  Pipeline_audit

In [0]:
%sql
SELECT *
FROM nutrichain_lakehouse.gold.pipeline_audit
ORDER BY run_ingested_at DESC
LIMIT 10;